In [3]:
# Cell 1
import os
import re
import time
import json
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv
from pypdf import PdfReader
from huggingface_hub import login

# Load environment variables
load_dotenv()

# Explicit HF Login to fix the unauthenticated warning
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    print("WARNING: HF_TOKEN not found in .env file.")

# Setup device
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Define paths
RAW_DATA_DIR = "../data/raw"
PROCESSED_DATA_DIR = "../data/processed"
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Using device: mps
PyTorch version: 2.11.0


In [4]:
# Cell 2
def clean_text(text):
    """Basic text cleaning to remove extra whitespace and fix PDF line breaks."""
    if not text:
        return ""
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_text_from_nested_pdfs(raw_dir):
    """Extracts text from PDFs across all subdirectories, grabbing metadata."""
    extracted_pages = []
    
    # Find all PDFs by walking through directories
    pdf_paths = []
    for root, _, files in os.walk(raw_dir):
        for file in files:
            if file.endswith('.pdf'):
                pdf_paths.append(os.path.join(root, file))
                
    if not pdf_paths:
        print(f"No PDFs found in {raw_dir} or its subfolders.")
        return []

    print(f"Found {len(pdf_paths)} PDF(s). Starting extraction...")
    start_time = time.time()
    
    # Outer progress bar for all PDF files
    for filepath in tqdm(pdf_paths, desc="Processing PDFs"):
        # Extract metadata from folder and file names
        filename = os.path.basename(filepath)
        folder_name = os.path.basename(os.path.dirname(filepath))
        
        # e.g., "Class_1_English" -> class_level="1", subject="English"
        parts = folder_name.split('_')
        class_level = parts[1] if len(parts) >= 2 else "Unknown"
        subject = parts[2] if len(parts) >= 3 else "Unknown"
        
        try:
            reader = PdfReader(filepath)
            # Inner progress bar for pages
            for page_num, page in enumerate(tqdm(reader.pages, desc=f"{filename}", leave=False)):
                text = page.extract_text()
                cleaned_text = clean_text(text)
                
                # Skip empty or nearly empty pages
                if len(cleaned_text) > 20: 
                    extracted_pages.append({
                        "filename": filename,
                        "class_level": class_level,
                        "subject": subject,
                        "page_num": page_num + 1,
                        "text": cleaned_text
                    })
        except Exception as e:
            print(f"Error processing {filename}: {e}")

    elapsed_time = time.time() - start_time
    print(f"Extraction complete! Extracted {len(extracted_pages)} valid pages in {elapsed_time:.2f} seconds.")
    
    return extracted_pages

# Run the extraction
extracted_data = extract_text_from_nested_pdfs(RAW_DATA_DIR)

Found 263 PDF(s). Starting extraction...


Processing PDFs:   0%|          | 0/263 [00:00<?, ?it/s]

bemr105.pdf:   0%|          | 0/5 [00:00<?, ?it/s]

bemr111.pdf:   0%|          | 0/8 [00:00<?, ?it/s]

bemr110.pdf:   0%|          | 0/8 [00:00<?, ?it/s]

...

ceev101.pdf:   0%|          | 0/18 [00:00<?, ?it/s]

ceev1ps.pdf:   0%|          | 0/16 [00:00<?, ?it/s]

ceev108.pdf:   0%|          | 0/9 [00:00<?, ?it/s]

ceev109.pdf:   0%|          | 0/12 [00:00<?, ?it/s]

Extraction complete! Extracted 3007 valid pages in 460.67 seconds.


In [5]:
# Cell 3
def create_overlapping_chunks(pages_data, chunk_size=200, overlap=20):
    """Splits text into word-based chunks with overlap."""
    chunks = []
    step_size = chunk_size - overlap
    
    print(f"Starting chunking (Size: {chunk_size} words, Overlap: {overlap} words)...")
    start_time = time.time()
    
    for page in tqdm(pages_data, desc="Chunking Pages"):
        words = page["text"].split()
        
        # Slide window across the words list
        for i in range(0, len(words), step_size):
            chunk_words = words[i : i + chunk_size]
            chunk_text = " ".join(chunk_words)
            
            # Only keep chunks with more than 10 words
            if len(chunk_words) > 10:
                chunks.append({
                    "filename": page["filename"],
                    "class_level": page["class_level"],
                    "subject": page["subject"],
                    "page_num": page["page_num"],
                    "chunk_text": chunk_text,
                    "word_count": len(chunk_words)
                })
                
    elapsed_time = time.time() - start_time
    print(f"Chunking complete! Created {len(chunks)} chunks in {elapsed_time:.2f} seconds.")
    
    return chunks

# Run the chunking
document_chunks = create_overlapping_chunks(extracted_data)

Starting chunking (Size: 200 words, Overlap: 20 words)...


Chunking Pages:   0%|          | 0/3007 [00:00<?, ?it/s]

Chunking complete! Created 3657 chunks in 0.03 seconds.


In [6]:
# Cell 4
OUTPUT_FILE = os.path.join(PROCESSED_DATA_DIR, "chunks.json")

print(f"Saving {len(document_chunks)} chunks to {OUTPUT_FILE}...")
start_time = time.time()

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(document_chunks, f, indent=4, ensure_ascii=False)

elapsed_time = time.time() - start_time
print(f"Saved successfully in {elapsed_time:.4f} seconds.")

Saving 3657 chunks to ../data/processed/chunks.json...
Saved successfully in 0.0371 seconds.


In [7]:
# Cell 5
if document_chunks:
    sample_idx = np.random.randint(0, len(document_chunks))
    sample = document_chunks[sample_idx]
    
    print("=== SANITY CHECK: RANDOM CHUNK ===")
    print(f"Class: {sample['class_level']} | Subject: {sample['subject']}")
    print(f"File: {sample['filename']} | Page: {sample['page_num']}")
    print(f"Word Count: {sample['word_count']}")
    print("-" * 50)
    print(sample['chunk_text'])
    print("-" * 50)
else:
    print("No chunks found!")

=== SANITY CHECK: RANDOM CHUNK ===
Class: 4 | Subject: English
File: desa109.pdf | Page: 13
Word Count: 92
--------------------------------------------------
101 Hekko A. You read how the people of Nagaland made the game of Hekko. Use the space given below to write about a new game that you would like to play. Let us Write Game: ........................................ Time: .................. Number of players: .......... Material needed: ......................................................................... Rules: .......................................................................................................... .......................................................................................................... .......................................................................................................... .......................................................................................................... ...............................................